In [ ]:
# =========================
# Imports
# =========================

# --- Standard library 
from datetime import datetime
import re
import json
import ast


# --- Third-party ---
from IPython.display import Markdown, display
from aisuite import Client

# --- Local / project ---
import research_tools

In [ ]:
import os
from dotenv import load_dotenv

# Clear existing environment key
os.environ.pop("ANTHROPIC_API_KEY", None)

# Reload only from .env
load_dotenv(override=True)

print("Now using:", os.getenv("ANTHROPIC_API_KEY")[:12] + "...")

In [ ]:
import os
from anthropic import Anthropic

client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

response = client.messages.create(
    model="claude-sonnet-4-5-20250929",
    max_tokens=100,
    messages=[
        {"role": "user", "content": "ping"}
    ]
)

print(response.content[0].text)

In [ ]:
CLIENT = Client()

# Starting Agents

In [ ]:
import ast
import re

def planner_agent(topic: str, model: str = "anthropic:claude-sonnet-4-5-20250929") -> list[str]:
    """
    Generates a plan as a Python list of steps (strings) for a research workflow.

    Args:
        topic (str): Research topic to investigate.
        model (str): Language model to use.

    Returns:
        List[str]: A list of executable step strings.
    """
    
    # Build the user prompt
    user_prompt = f"""
    You are a planning agent responsible for organizing a research workflow with multiple intelligent agents.

    🧠 Available agents:
    - A research agent who can search the web, Wikipedia,pubmed and arXiv.
    - A writer agent who can draft research summaries.
    - An editor agent who can reflect and revise the drafts.

    🎯 Your job is to write a clear, step-by-step research plan **as a valid Python list**, where each step is a string.
    Each step should be atomic, executable, and must rely only on the capabilities of the above agents.
    - Return a clear, step by step research plan as a valid Pythin list where each step is a string.
    - Avoid vague steps like "do more research"- be more specific about quieries, outputs anc checls
    
    
    🚫 DO NOT include irrelevant tasks like "create CSV", "set up a repo", "install packages", etc.
    ✅ DO include real research-related tasks (e.g., search, summarize, draft, revise).
    ✅ DO assume tool use is available.
    ✅ DO NOT include explanation text — return ONLY the Python list.
    ✅ The final step should be to generate a Markdown document containing the complete research report.

    Topic: "{topic}"
    """

    # Add the user prompt to the messages list
    messages = [{"role": "user", "content": user_prompt}]

    # Call the LLM
    response = CLIENT.chat.completions.create( 
        # Pass in the model
        model=model,
        # Define the messages. Remember this is meant to be a user prompt!
        messages=messages,
        # Keep responses creative
        temperature=1, 
    )

    steps_str = response.choices[0].message.content.strip()

    # Remove markdown fences if the model adds them anyway
    steps_str = re.sub(r"^```(?:python)?\s*", "", steps_str)
    steps_str = re.sub(r"\s*```$", "", steps_str).strip()

    # Parse safely
    steps = ast.literal_eval(steps_str)

    # Validate output
    if not isinstance(steps, list):
        raise ValueError(f"Planner output is not a list: {steps}")
    if not all(isinstance(step, str) for step in steps):
        raise ValueError(f"Planner output must be a list of strings: {steps}")


    return steps

In [ ]:
def research_agent(task: str, model: str = "anthropic:claude-sonnet-4-5-20250929", return_messages: bool = False):
    """
    Executes a research task using tools via aisuite (no manual loop).
    Returns either the assistant text, or (text, messages) if return_messages=True.
    """
    print("==================================")  
    print("🔍 Research Agent")                 
    print("==================================")

    current_time = datetime.now().strftime('%Y-%m-%d')

    prompt = f"""
    
    You are an advanced research assistant
    
    Available tools:
    - arxiv_tool (search scientific papers)
    - tavily_tool (web search)
    - wikipedia_tool (encyclopedic summaries)
    - pubmed_tool (search biomedical papers)
    
    Task:
    {task}
    
    Guidelines:
    - Use tools when necessary.
    - Cite sources clearly with URLs.
    - Use an academic tone.
    - Provide a structured, well-organised research summary.
    
    """
    
    # Create the messages dict to pass to the LLM. Remember this is a user prompt!
    messages = [{"role": "user", "content": prompt}]

    # Save all of your available tools in the tools list. These can be found in the research_tools module.
    # You can identify each tool in your list like this: 
    # research_tools.<name_of_tool>, where <name_of_tool> is replaced with the function name of the tool.
    tools = [
        research_tools.arxiv_search_tool,
        research_tools.tavily_search_tool,
        research_tools.wikipedia_search_tool,
        research_tools.pubmed_search_tool
        
    ]
    
    # Call the model with tools enabled
    response = CLIENT.chat.completions.create(  
        # Set the model
        model=model,
        # Pass in the messages. You already defined this!
        messages=messages,
        # Pass in the tools list. You already defined this!
        tools=tools,
        # Set the LLM to automatically choose the tools
        tool_choice="auto",
        # Set the max turns to 6
        max_turns=2
    )  
    

    content = response.choices[0].message.content
    print("✅ Output:\n", content)

    
    return (content, messages) if return_messages else content  

In [ ]:
def writer_agent(task: str, model: str = "anthropic:claude-sonnet-4-5-20250929") -> str: # @REPLACE def writer_agent(task: str, model: str = None) -> str:
    """
    Executes writing tasks, such as drafting, expanding, or summarizing text.
    """
    print("==================================")
    print("✍️ Writer Agent")
    print("==================================")

    ### START CODE HERE ###
    
    # Create the system prompt.
    # This should assign the LLM the role of a writing agent specialized in generating well-structured academic or technical content
    system_prompt = f"""
    
    You are a writing agent that focuses on generating academic or technical content
   
   Guidelines:
   -Use clear section headings
   -Ensure logical flow and coherence
   -Avoid filler langauge
   -Provide structured paragraphs
   """


    # Define the system msg by using the system_prompt and assigning the role of system
    system_msg = {"role":"system", "content":system_prompt}

    # Define the user msg. In this case the user prompt should be the task passed to the function
    user_msg = {"role":"user", "content":task}

    # Add both system and user messages to the messages list
    messages = [system_msg, user_msg]
    

    response = CLIENT.chat.completions.create(
        model=model, 
        messages=messages,
        temperature=1.0
    )

    return response.choices[0].message.content

In [ ]:
def editor_agent(task: str, model: str = "anthropic:claude-sonnet-4-5-20250929") -> str:
    """
    Executes editorial tasks such as reflection, critique, or revision.
    """
    print("==================================")
    print("🧠 Editor Agent")
    print("==================================")
    
    system_prompt = f"""
    
    You are an editor agent whose task is to reflect on, critque and improve drafts
    
    Guidelines:
   -Use clear section headings
   -Ensure logical flow and coherence
   -Avoid filler langauge
   -Provide structured paragraphs
   
    """
    
    # Define the system msg by using the system_prompt and assigning the role of system
    system_msg = {"role":"system", "content":system_prompt}
    
    # Define the user msg. In this case the user prompt should be the task passed to the function
    user_msg = {"role":"user","content":task}
    
    # Add both system and user messages to the messages list
    messages = [system_msg, user_msg]
    
    response = CLIENT.chat.completions.create(
        model=model, 
        messages=messages,
        temperature=0.7 
    )
    
    return response.choices[0].message.content

In [ ]:
def summarising_agent(task: str, model: str = "anthropic:claude-sonnet-4-5-20250929") -> str:
    """
    Produces a structured, concise summary of a research paper or draft.
    """
    print("==================================")
    print("🧠 Summarising Agent")
    print("==================================")
    
    system_prompt = f"""
    
    You are a research summarisation agent.

    Your task is to produce a clear, concise, and structured summary of the provided content.

    Guidelines:
    - Extract only the most important information.
    - Preserve factual accuracy.
    - Do NOT introduce new information.
    - Do NOT critique or rewrite beyond summarising.
    - Use clear section headings:
        • Key Findings
        • Conclusions
    - Keep the summary concise and focused.
    """

    # Define the system msg by using the system_prompt and assigning the role of system
    system_msg = {"role":"system", "content":system_prompt}
    
    # Define the user msg. In this case the user prompt should be the task passed to the function
    user_msg = {"role":"user","content":task}
    
    # Add both system and user messages to the messages list
    messages = [system_msg, user_msg]
    
    response = CLIENT.chat.completions.create(
        model=model, 
        messages=messages,
        temperature=0.3 
    )
    
    return response.choices[0].message.content

In [ ]:
import json

def final_summary_agent(task: str, model: str = "anthropic:claude-sonnet-4-5-20250929") -> str:
    """
    Produces a final structured summary for a T2DM hypothesis,
    based only on the evidence provided in the task/context.
    """
    print("==================================")
    print("🧾 Final Summary Agent")
    print("==================================")

    system_prompt = """
You are a final summary agent for a medical hypothesis evaluation workflow.

Your role is to read the provided research/context and produce a final, structured conclusion
about whether the hypothesis meaningfully affects predicted Type 2 Diabetes Mellitus (T2DM) likelihood or severity.

You must base your answer ONLY on the provided evidence.
Do not invent facts, numbers, or effects that are not stated or strongly implied.
If evidence is weak, inconsistent, or missing, say so clearly.

Return the answer in exactly this format:

In the Anthropic model,

(<hypothesis>) is <description1> to change the predicted T2DM severity.

The first variable showed:
<brief summary of the first variable's individual effect>

The second variable showed:
<brief summary of the second variable's individual effect>

The combination of the variables showed:
<brief summary of the combined effect>

Therefore, this hypothesis is <description2> to meaningfully affect likelihood of T2DM.

Prediction certainty: <High / Moderate / Low>
Model reliability: <High / Moderate / Low>

Rules:
- Keep the wording cautious and evidence-based.
- Keep each section concise.
- "description1" should be one of:
  likely, somewhat likely, unlikely, not clearly likely, plausibly associated
- "description2" should be one of:
  likely, moderately supported, weakly supported, unlikely, insufficiently supported
- "Prediction certainty" reflects how clear and consistent the conclusion is from the provided evidence.
- "Model reliability" reflects how dependable the underlying evidence/context appears.
- If one variable has much stronger evidence than the other, say that explicitly.
- If the combination effect is mostly driven by one variable, say so.
"""

    response = CLIENT.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": task}
        ],
        temperature=0.2
    )

    return response.choices[0].message.content

In [ ]:
agent_registry = {
    "research_agent": research_agent,
    "editor_agent": editor_agent,
    "writer_agent": writer_agent,
    "summarising_agent": summarising_agent,
    "final_summary_agent": final_summary_agent
}

def clean_json_block(raw: str) -> str:
    """
    Clean the contents of a JSON block that may come wrapped with Markdown backticks.
    """
    raw = raw.strip()
    if raw.startswith("```"):
        raw = re.sub(r"^```(?:json)?\n?", "", raw)
        raw = re.sub(r"\n?```$", "", raw)
    return raw.strip()

In [ ]:
def executor_agent(topic, model: str = "anthropic:claude-sonnet-4-5-20250929", limit_steps: bool = True):

    plan_steps = planner_agent(topic)

    max_steps = 4   # ← add this back

    if limit_steps:
        plan_steps = plan_steps[:max_steps]

    plan_steps.append(
        "Summarise all findings from previous steps into a concise synthesis with key quantitative takeaways and limitations"
    )

    history = []

    print("==================================")
    print("🎯 Editor Agent")
    print("==================================")

    for i, step in enumerate(plan_steps):

        agent_decision_prompt = f"""
        You are an execution manager for a multi-agent research team.

        Given the following instruction, identify which agent should perform it and extract the clean task.

        Return only a valid JSON object with two keys:
        - "agent": one of ["research_agent", "editor_agent", "writer_agent", "summarising_agent"]
        - "task": a string with the instruction that the agent should follow

        Only respond with a valid JSON object. Do not include explanations or markdown formatting.

        Instruction: "{step}"
        """
        response = CLIENT.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": agent_decision_prompt}],
            temperature=0,
        )

        raw_content = response.choices[0].message.content
        cleaned_json = clean_json_block(raw_content)
        agent_info = json.loads(cleaned_json)

        agent_name = agent_info["agent"]
        task = agent_info["task"]

        context = "\n".join([
            f"Step {j+1} executed by {a}:\n{r}" 
            for j, (s, a, r) in enumerate(history)
        ])
        enriched_task = f"""
        You are {agent_name}.

        Here is the context of what has been done so far:
        {context}

        Your next task is:
        {task}
        """

        print(f"\n🛠️ Executing with agent: `{agent_name}` on task: {task}")

        if agent_name in agent_registry:
            output = agent_registry[agent_name](enriched_task)
            history.append((step, agent_name, output))
        else:
            output = f"⚠️ Unknown agent: {agent_name}"
            history.append((step, agent_name, output))

        print(f"✅ Output:\n{output}")

    return history

In [ ]:
print(question_vars)

In [ ]:
from IPython.display import Markdown, display
import pandas as pd

def make_question_text(r):
    left = f"{r['level1']} {r['var1']}" if r["level1"] is not None else r["var1"]
    right = f"{r['level2']} {r['var2']}" if r["level2"] is not None else r["var2"]
    return f"Does {left} AND {right} affect your likelihood of Type 2 Diabetes?"

question_list = [make_question_text(results[0])]

executor_runs = []

for idx, question_vars in enumerate(question_list, start=1):
    print("\n" + "=" * 70)
    print(f"Running executor_agent for Question {idx}")
    print(question_vars)
    print("=" * 70)

    executor_history = executor_agent(question_vars, limit_steps=True)
    md = executor_history[-1][-1].strip("`")

    executor_runs.append({
        "question_num": idx,
        "question_vars": question_vars,
        "executor_history": executor_history,
        "final_markdown": md
    })

for run in executor_runs:
    print("\n" + "=" * 70)
    print(f"Final output for Question {run['question_num']}")
    print(run["question_vars"])
    print("=" * 70)
    display(Markdown(run["final_markdown"]))

executor_summary_df = pd.DataFrame([
    {
        "question_num": run["question_num"],
        "question_vars": run["question_vars"],
        "final_markdown": run["final_markdown"]
    }
    for run in executor_runs
])

executor_summary_df.to_csv("executor_5_question_outputs.csv", index=False)

# Summary

In [ ]:
from IPython.display import Markdown, display
import pandas as pd

def build_summary_context(history, max_chars=4000):
    parts = []
    for i, (step, agent, output) in enumerate(history):
        parts.append(
            f"Step {i+1}\n"
            f"Output: {str(output)[:800]}"
        )
    text = "\n\n".join(parts)
    return text[-max_chars:]


def make_question_text(r):
    left = f"{r['level1']} {r['var1']}" if r["level1"] is not None else r["var1"]
    right = f"{r['level2']} {r['var2']}" if r["level2"] is not None else r["var2"]
    return f"Does {left} AND {right} affect your likelihood of Type 2 Diabetes?"

In [ ]:
from IPython.display import Markdown, display

all_final_outputs = []

for run in executor_runs:
    hypothesis = run["question_vars"]
    executor_history = run["executor_history"]

    summary_context = build_summary_context(executor_history)

    final_task = f"""
Hypothesis:
{hypothesis}

Context from previous agent outputs:
{summary_context}

Produce a final answer in exactly this format:

In the Anthropic model,

({hypothesis}) is <description1> to change the predicted T2DM severity.

The first variable showed:
<summary>

The second variable showed:
<summary>

The combination of the variables showed:
<summary>

Therefore, this hypothesis is <description2> to meaningfully affect likelihood of T2DM.

Prediction certainty: <High / Moderate / Low>

Model reliability: <High / Moderate / Low>
"""

    final_output = final_summary_agent(final_task)

    all_final_outputs.append({
        "question_num": run["question_num"],
        "hypothesis": hypothesis,
        "final_output": final_output
    })

    print("\n" + "=" * 70)
    print(f"Question {run['question_num']}")
    print(hypothesis)
    print("=" * 70)
    display(Markdown(final_output))